### Import Required Libraries and Modules

In [2]:
import h5py
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from Architectures.perceiver import Perceiver

from sklearn.metrics import r2_score

### DataSet Reading and Creation
*"data" : (16377, 8) : RNA Half Life* <br>
*"label" : (16377, ) : RNA Expression level* <br>
*"promoter" : (16377, 20000, 4) : DNA Promoter (One-Hot Encoded)* <br>
*"promoter_numeric" : (16377, 20000) : DNA Promoter (Numeric)*

## Plotting Data

In [3]:
file = h5py.File('Data/train.h5', 'r')
print("Available Keys: ", list(file.keys()))

dataset_data = file['data']
half_life = dataset_data[:] # RNA Half Life

dataset_label = file['label']
label = dataset_label[:]    # Output Label: Rna Expression Level

dataset_promoter = file['promoter']
promoter = dataset_promoter[:]  # Influencing DNA

# print(data)
file.close()

Available Keys:  ['data', 'geneName', 'label', 'promoter']


In [ ]:
promoter_numeric = np.argmax(promoter, axis=-1)
promoter_numeric = promoter_numeric[:10]
print(promoter_numeric.shape)
print(promoter_numeric[0])

(10, 20000)
[1 3 0 ... 0 0 3]


## Training

### Model Definition

In [ ]:
# TODO Reconsider all these, and consider for protein sequence as well
class config:
    embed_dim = 256
    model_dim = 64
    num_heads = 8 # embed_dim % num_heads == 0
    output_dim = 1
    vocab_size = 4  # A, C, G, T > 4
    max_position_embeddings = 20048
    dropout_prob = 0.1
    perciever_decoder_num_layers = 10
    num_convo_attention_layers = 10       # N Layers
    conv_in_convo_attention = 32
    conv_out_convo_attention = 32


In [ ]:
class ConvoAttentionConfig:
    def __init__(self):
        self.in_channels = 4  # Example: RGB image input
        self.out_channels = 64
        self.kernel_size = 3
        self.pool_kernel_size = 1
        self.stride = 1
        self.pool_stride = 1
        self.embed_dim = 4  # Should match out_channels for compatibility
        self.model_dim = 4   # Dimension used in attention mechanism
        self.num_convo_attention_layers = 3

In [ ]:
class Conv1DAttentionConfig:
    def __init__(self):
        self.in_channels=4
        self.out_channels=64
        self.kernel_size=3
        self.stride=1
        self.pool_kernel_size=2
        self.pool_stride=2
        self.embed_dim=4
        self.model_dim=4

In [ ]:
Conv1DAttentionConfig().kernel_size

3

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Perceiver(config(), ConvoAttentionConfig(), Conv1DAttentionConfig()).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

### Data-Set Creation

In [ ]:
X_train_Half_life = torch.tensor(half_life, dtype=torch.float32).to(device)
X_train_DNA_numeric= torch.tensor(promoter_numeric, dtype=torch.float32).to(device)
X_train_DNA_one_hot= torch.tensor(promoter, dtype=torch.float32).to(device)
Y_train_label = torch.tensor(label.reshape(-1, 1), dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_Half_life, X_train_DNA_numeric, X_train_DNA_one_hot , Y_train_label)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)  

### Train

In [ ]:
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    total_batches = len(train_loader)
    print(f"Total batches: {total_batches}")

    for batch_idx, (half_life_batch, promoter_numeric_batch, promoter_one_hot_batch, label_batch) in enumerate(train_loader):
        if batch_idx % 50 == 0:
            print(f"Step {batch_idx}/{total_batches}")

        optimizer.zero_grad()

        # Ensure input tensors for embedding layers are of type LongTensor
        half_life_batch = half_life_batch.to(device).float()
        promoter_numeric_batch = promoter_numeric_batch.to(device).long()
        promoter_one_hot_batch = promoter_one_hot_batch.to(device).float()
        label_batch = label_batch.to(device).float()

        # Forward pass
        outputs = model(half_life_batch, promoter_numeric_batch, promoter_one_hot_batch)
        loss = loss_fn(outputs, label_batch)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()  # Accumulate the loss for this batch

    print(f"Epoch [{epoch+1}/{num_epochs}]--------------------Loss: [{epoch_loss / len(train_loader)}]------------Accuracy: [{evaluate(outputs, label_batch)}]")

Total batches: 1
Step 0/1


### Test

In [ ]:
from sklearn.metrics import r2_score

def evaluate(y_pred, y_test):
    y_pred = y_pred.detach().cpu().numpy()
    y_test = y_test.detach().cpu().numpy()
    return r2_score(y_test, y_pred)

In [ ]:
model.eval()
outputs = model(test_loader[""])